# WF-001 Stage-1 Demo (Pure Python)

This notebook is a **pure-Python** step-by-step demonstration of the WF-001 stage-1 pipeline.

Instead of calling the CLI via subprocess, it:

- adds `src/` to `sys.path`
- loads the YAML config via `rpbench.config.load_config`
- calls `rpbench.runners.run_demo.run_demo(cfg)` directly
- writes outputs under the repo’s `data/` directory

> The canonical path is still the CLI (`scripts/run_demo.py`), but this notebook shows the importable API path.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
SRC_DIR = REPO_ROOT / 'src'
DATA_DIR = REPO_ROOT / 'data'
RUNS_DIR = DATA_DIR / 'outputs' / 'runs'
REPORT_DIR = DATA_DIR / 'outputs' / 'reports'

# Make `import rpbench` work in-notebook without installing as a package
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Repo root:', REPO_ROOT)
print('src dir:', SRC_DIR)
print('data dir:', DATA_DIR)
print('runs dir:', RUNS_DIR)
print('report dir:', REPORT_DIR)


Repo root: /home/wei402/Desktop/rp-benchmark
src dir: /home/wei402/Desktop/rp-benchmark/src
data dir: /home/wei402/Desktop/rp-benchmark/data
runs dir: /home/wei402/Desktop/rp-benchmark/data/outputs/runs
report dir: /home/wei402/Desktop/rp-benchmark/data/outputs/reports


In [2]:
from rpbench.config import load_config
from rpbench.runners.run_demo import run_demo
from rpbench.utils.io import save_jsonl

cfg = load_config(REPO_ROOT / 'configs' / 'demo' / 'wf001_demo_autompg.yaml')
cfg.output_root = str(RUNS_DIR)

records = run_demo(cfg)

out_path = Path(cfg.output_root) / 'demo_results.jsonl'
save_jsonl(records, out_path)

print(f'Saved {len(records)} records to {out_path}')


  [1/40] Mech_RP eps=0.5 seed=0
  [2/40] Mech_RP eps=0.5 seed=1
  [3/40] Mech_RP eps=0.5 seed=2
  [4/40] Mech_RP eps=0.5 seed=3
  [5/40] Mech_RP eps=0.5 seed=4
  [6/40] Mech_RP eps=1.0 seed=0
  [7/40] Mech_RP eps=1.0 seed=1
  [8/40] Mech_RP eps=1.0 seed=2
  [9/40] Mech_RP eps=1.0 seed=3
  [10/40] Mech_RP eps=1.0 seed=4
  [11/40] Mech_RP eps=2.0 seed=0
  [12/40] Mech_RP eps=2.0 seed=1
  [13/40] Mech_RP eps=2.0 seed=2
  [14/40] Mech_RP eps=2.0 seed=3
  [15/40] Mech_RP eps=2.0 seed=4
  [16/40] Mech_RP eps=4.0 seed=0
  [17/40] Mech_RP eps=4.0 seed=1
  [18/40] Mech_RP eps=4.0 seed=2
  [19/40] Mech_RP eps=4.0 seed=3
  [20/40] Mech_RP eps=4.0 seed=4
  [21/40] Blocki12_JL eps=0.5 seed=0
  [22/40] Blocki12_JL eps=0.5 seed=1
  [23/40] Blocki12_JL eps=0.5 seed=2
  [24/40] Blocki12_JL eps=0.5 seed=3
  [25/40] Blocki12_JL eps=0.5 seed=4
  [26/40] Blocki12_JL eps=1.0 seed=0
  [27/40] Blocki12_JL eps=1.0 seed=1
  [28/40] Blocki12_JL eps=1.0 seed=2
  [29/40] Blocki12_JL eps=1.0 seed=3
  [30/40] Blocki

In [3]:
from rpbench.reporting.tables import build_summary_table
from rpbench.reporting.figures import plot_eps_vs_mse
from rpbench.reporting.summary import write_summary
from rpbench.utils.io import load_jsonl

input_path = RUNS_DIR / 'demo_results.jsonl'
rows = load_jsonl(input_path)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = REPORT_DIR / 'summary_table.csv'
fig_path = REPORT_DIR / 'eps_vs_test_mse.png'
md_path = REPORT_DIR / 'demo_summary.md'

build_summary_table(rows, csv_path)
plot_eps_vs_mse(rows, fig_path)
write_summary(rows, md_path, fig_path.name, csv_path.name)

print('Wrote:')
print(' -', csv_path)
print(' -', fig_path)
print(' -', md_path)

Wrote:
 - /home/wei402/Desktop/rp-benchmark/data/outputs/reports/summary_table.csv
 - /home/wei402/Desktop/rp-benchmark/data/outputs/reports/eps_vs_test_mse.png
 - /home/wei402/Desktop/rp-benchmark/data/outputs/reports/demo_summary.md


In [4]:
import pandas as pd

summary_csv = REPORT_DIR / 'summary_table.csv'
summary_md = REPORT_DIR / 'demo_summary.md'
fig_png = REPORT_DIR / 'eps_vs_test_mse.png'

display(pd.read_csv(summary_csv))

print('\n--- demo_summary.md ---\n')
print(summary_md.read_text())

print('\nFigure path:', fig_png)


,mechanism,epsilon,test_mse_mean,test_mse_std,rel_fro_mean,rel_fro_std,runtime_mean,n_seeds
0,Blocki12_JL,0.5,1.413302,1.112051,2.363386,0.486530,0.001287,5
1,Blocki12_JL,1.0,65.279586,103.490432,0.721070,0.142889,0.001260,5
2,Blocki12_JL,2.0,1.111916,1.259951,0.312035,0.062797,0.001258,5
3,Blocki12_JL,4.0,0.699525,1.051595,0.210142,0.050451,0.001257,5
4,Mech_RP,0.5,231.575043,373.524653,0.958898,0.217304,0.000563,5
5,Mech_RP,1.0,6.392646,12.194454,0.658745,0.160866,0.000538,5
6,Mech_RP,2.0,14.506209,30.798966,0.500293,0.129049,0.000530,5
7,Mech_RP,4.0,1.067987,0.497648,0.412872,0.113534,0.000532,5



--- demo_summary.md ---

# WF-001 Demo Run Summary

**Dataset:** autompg  
**Mechanisms:** Blocki12_JL, Mech_RP  
**Task:** OLSFromRelease  
**Epsilon grid:** [0.5, 1.0, 2.0, 4.0]  
**Delta:** 1.13e-05  
**Seeds:** [0, 1, 2, 3, 4]  
**Total records:** 41  

**Non-private baseline test MSE:** 0.147279

## Results

See [summary_table.csv](summary_table.csv) for the aggregate table.

![Privacy-Utility Curve](eps_vs_test_mse.png)

## Outputs

- Row-level results: `data/outputs/runs/demo_results.jsonl`
- Summary table: `summary_table.csv`
- Figure: `eps_vs_test_mse.png`


Figure path: /home/wei402/Desktop/rp-benchmark/data/outputs/reports/eps_vs_test_mse.png
